# Spin-up Quick Check Notebook
This notebook is organized for spin-up diagnostics on monthly model output.

## Task 1: Domain map animation of a selected biological variable
- Input file: `dws_500m.3d.201501.nc`
- Goal: animate a selected variable over time on the model map
- Output: an animated GIF (optional) and inline animation preview

## Task 2: Aggregated time series of a selected biological variable for the whole domain
- Cell 2 (Task 2): build and plot the domain-mean trend at a selected layer (4d) or 3d variable
- Input file: `dws_500m.3d.201501.nc`
- Goal: visualize time series of spatially averaged values over the whole domain also for a specific vertical level;
- Output: a time-series line plot

In [ ]:
# meta data:
#! BFM biological model

# pelagic variables (group PelVariables):
#! pelagic  (O)              O2:   Oxygen (mmol/m3)
#! pelagic  (P)              N1:   Phosphate (mmol/m3)
#! pelagic  (N)              N3:   Nitrate (mmol/m3)
#! pelagic  (N)              N4:   Ammonium (mmol/m3)
#! pelagic  (Si)             N5:   Silicate (mmol/m3)
#! pelagic  (R)              N6:   Reduction Equivalents (mmol/m3)
#! pelagic  (N)              O4:   N2-sink (mmol/m3)
#! pelagic  (CNP)            B1:   Pelagic Bacteria
#! pelagic  (CNPSiI)         P1:   Diatoms (group PhytoPlankton))
#! pelagic  (CNPSiI)         P2:   Flagellates (group PhytoPlankton))
#! pelagic  (CNPSiI)         P3:   PicoPhytoPlankton (group PhytoPlankton))
#! pelagic  (CNPSiI)         P4:   Dinoflagellates (group PhytoPlankton))
#! pelagic  (CNP)            Z3:   Carnivorous mesozooplankton (group MesoZooPlankton))
#! pelagic  (CNP)            Z4:   Omnivorous mesozooplankton (group MesoZooPlankton))
#! pelagic  (CNP)            Z5:   Microzooplankton (group MicroZooPlankton))
#! pelagic  (CNP)            Z6:   Heterotrophic nanoflagellates (HNAN) (group MicroZooPlankton))
#! pelagic  (CNPSi)          R1:   Labile Organic Carbon (LOC)
#! pelagic  (C)              R2:   CarboHydrates (sugars)
#! pelagic  (CNPSi)          R6:   Particulate Organic Carbon (POC)
#! pelagic  (C)              R7:   Refractory Disoolved Organic Carbon

# Benthic variables (group BenVariables):
# ! benthic  (CNP)            Y1:   Epibenthos (group BenOrganisms))
# ! benthic  (CNP)            Y2:   Deposit feeders (group BenOrganisms))
# ! benthic  (CNP)            Y3:   Suspension feeders (group BenOrganisms))
# ! benthic  (CNP)            Y4:   Meiobenthos (group ! BenOrganisms))
# ! benthic  (CNP)            Y5:   Benthic predators (group BenOrganisms))
# ! benthic  (CNPSi)          Q1:   Labile organic carbon (group BenDetritus))
# ! benthic  (CNPSi)          Q11:  Labile organic carbon (group BenDetritus))
# ! benthic  (CNPSi)          Q6:   Particulate organic carbon (group BenDetritus))
# ! benthic  (CNP)            H1:   Aerobic benthic bacteria (group BenBacteria))
# ! benthic  (CNP)            H2:   Anaerobic benthic bacteria (group BenBacteria))
# ! benthic  (P)              K1:   Phosphate in oxic layer (group BenthicPhosphate))
# ! benthic  (P)              K11:  Phosphate in denit layer (group BenthicPhosphate))
# ! benthic  (P)              K21:  Phosphate in anoxic layer (group BenthicPhosphate))
# ! benthic  (N)              K4:   Ammonium in oxic layer (group BenthicAmmonium))
# ! benthic  (N)              K14:  Ammonium in denit layer (group BenthicAmmonium))
# ! benthic  (N)              K24:  Ammonium in anoxic layer (group BenthicAmmonium))
# ! benthic  (R)              K6:   Reduction equivalents 
# ! benthic  (M)              D1:   Oxygen penetration depth
# ! benthic  (M)              D2:   Denitrification depth 
# ! benthic  (M)              D6:   Depth distribution factor organic C 
# ! benthic  (M)              D7:   Depth distribution factor organic N
# ! benthic  (M)              D8:   Depth distribution factor organic P
# ! benthic  (M)              D9:   Depth distribution factor organic Si
# ! benthic  (O)              G2:   Benthic O2



# Jetty dataset meta data:
# TSM: mg/L
# C: mg/m3
# TOC: mgC/L
# POC: mgC/L
# DOC: mgC/L

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import contextily as ctx

# Setup
DATA_DIR = Path('/export/lv9/projects/dws/results/validation/benthic_diatom')
data_path = DATA_DIR / 'SIBES_Diatom_2024.csv'

BASEMAP = ctx.providers.Esri.WorldShadedRelief

def dms_to_dd(dms_str):
    """Convert DMS string e.g. 53°23'34.8''N to decimal degrees."""
    match = re.match(r"(\d+)°(\d+)'([\d.]+)''([NSEW])", str(dms_str).strip())
    deg, minutes, sec, direction = match.groups()
    dd = float(deg) + float(minutes) / 60 + float(sec) / 3600
    if direction in ('S', 'W'):
        dd = -dd
    return dd

# Load data
df_2024 = pd.read_csv(data_path, encoding="latin-1")
df_2024["lat_dd"] = df_2024["latitude"].apply(dms_to_dd)
df_2024["lon_dd"] = df_2024["longitude"].apply(dms_to_dd)

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(df_2024["lon_dd"], df_2024["lat_dd"], c=df_2024["chla_ug_m2"], cmap="plasma", s=30, edgecolors="none", zorder=2)
plt.colorbar(sc, ax=ax, label="chlorophyll-a ug/m2")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Benthic Chla - 2024")

# Add basemap — must come after scatter so ax has a valid extent
ctx.add_basemap(ax, crs="EPSG:4326", source=BASEMAP, zorder=1)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import contextily as ctx

# Setup
DATA_DIR = Path('/export/lv9/projects/dws/results/validation/benthic_diatom')
data_path = DATA_DIR / 'diatoms_2014.csv'

BASEMAP = ctx.providers.Esri.WorldShadedRelief

# Load data
df = pd.read_csv(data_path)

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(df["X"], df["Y"], c=df["Diatom1"], cmap="plasma", s=30, edgecolors="none", zorder=2)
plt.colorbar(sc, ax=ax, label="diatom1")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Diatom 1 - 2014")

# Add basemap — must come after scatter so ax has a valid extent
ctx.add_basemap(ax, crs="EPSG:4326", source=BASEMAP, zorder=1)

plt.tight_layout()
plt.show()

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import contextily as ctx

# Setup
DATA_DIR = Path('/export/lv9/projects/dws/results/validation/benthic_diatom')
BASEMAP = ctx.providers.Esri.WorldShadedRelief

def dms_to_dd(dms_str):
    """Convert DMS string e.g. 53°23'34.8''N to decimal degrees."""
    match = re.match(r"(\d+)°(\d+)'([\d.]+)''([NSEW])", str(dms_str).strip())
    deg, minutes, sec, direction = match.groups()
    dd = float(deg) + float(minutes) / 60 + float(sec) / 3600
    if direction in ('S', 'W'):
        dd = -dd
    return dd

# Load 2014 data (decimal degrees)
# unit seems to be ug/cm2 → convert to µg/m²
df_2014 = pd.read_csv(DATA_DIR / 'diatoms_2014.csv')
df_2014["avgPhyto"] = (df_2014["avgDiatom"] + df_2014["avgCyano"] + df_2014["avgGreen"]) * 1e4

# Load 2024 data (DMS → decimal degrees)
df_2024 = pd.read_csv(DATA_DIR / 'SIBES_Diatom_2024.csv', encoding="latin-1")
df_2024["lat_dd"] = df_2024["latitude"].apply(dms_to_dd)
df_2024["lon_dd"] = df_2024["longitude"].apply(dms_to_dd)

# Shared colour limits
vmin, vmax = 0, 70_000
cmap = "plasma"          # same colormap for both

# Plot
fig, ax = plt.subplots(figsize=(10, 8))

sc1 = ax.scatter(
    df_2014["X"], df_2014["Y"],
    c=df_2014["avgPhyto"],
    cmap=cmap, vmin=vmin, vmax=vmax,
    s=30, edgecolors="none", zorder=2,
    label="Sleuter PhytoBen 2014"
)

sc2 = ax.scatter(
    df_2024["lon_dd"], df_2024["lat_dd"],
    c=df_2024["chla_ug_m2"],
    cmap=cmap, vmin=vmin, vmax=vmax,
    s=30, marker="^", edgecolors="none", zorder=3,
    label="SIBES PhytoBen 2024"
)

# Single shared colour bar
cbar = plt.colorbar(sc1, ax=ax, fraction=0.03, pad=0.04)
cbar.set_label("Concentration (µg C m⁻²)")

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Benthic Diatoms 2014 & 2024")
ax.legend(loc="upper left")

# Basemap — must come after scatter so ax has a valid extent
ctx.add_basemap(ax, crs="EPSG:4326", source=BASEMAP, zorder=1)

plt.tight_layout()
plt.show()

In [ ]:
# Compare 2014 & 2024 field data with modelled benthic concentrations
# (nearest 4 model cells, August only)

import re
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

# ------------------------------------------------------------------
# 1. Field data
# ------------------------------------------------------------------
DATA_DIR = Path('/export/lv9/projects/dws/results/validation/benthic_diatom')

def dms_to_dd(dms_str):
    """Convert DMS string e.g. 53°23'34.8''N to decimal degrees."""
    match = re.match(r"(\d+)°(\d+)'([\d.]+)''([NSEW])", str(dms_str).strip())
    if match is None:
        raise ValueError(f"Could not parse DMS: {dms_str}")
    deg, minutes, sec, direction = match.groups()
    dd = float(deg) + float(minutes) / 60 + float(sec) / 3600
    if direction in ('S', 'W'):
        dd = -dd
    return dd

# --- 2024 SIBES ---
df_2024 = pd.read_csv(DATA_DIR / 'SIBES_Diatom_2024.csv', encoding='latin-1')
df_2024['lat'] = df_2024['latitude'].apply(dms_to_dd)
df_2024['lon'] = df_2024['longitude'].apply(dms_to_dd)

Chla_Carbon_Conversion_Factor = 0.025
df_2024['carbon_mg_m2'] = df_2024['chla_ug_m2'] / Chla_Carbon_Conversion_Factor / 1000

# --- 2014 ---
df_2014 = pd.read_csv(DATA_DIR / 'diatoms_2014.csv')
# original unit appears to be µg cm⁻² → convert to µg m⁻² then to carbon
df_2014['avgPhyto_ug_m2'] = (df_2014['avgDiatom'] + df_2014['avgCyano'] + df_2014['avgGreen']) * 1e4
df_2014['carbon_mg_m2'] = df_2014['avgPhyto_ug_m2'] / Chla_Carbon_Conversion_Factor / 1000
df_2014['lon'] = df_2014['X']
df_2014['lat'] = df_2014['Y']

# ------------------------------------------------------------------
# 2. Model data – August only
# ------------------------------------------------------------------
MODEL_DIR = Path('/export/lv9/projects/dws/model_output/archived_runs/spinup_10/')
FILE_PATTERN = 'dws_500m.3d.2015??.nc'

model_files = sorted(MODEL_DIR.glob(FILE_PATTERN))
if not model_files:
    raise FileNotFoundError(f'No files matching {FILE_PATTERN} in {MODEL_DIR}')

ds = xr.open_mfdataset(
    model_files,
    combine='nested',
    concat_dim='time',
    decode_times=True,
    data_vars='minimal',
    coords='minimal',
    compat='override',
    join='override',
)

VAR_NAME = 'BP1c'          # <-- change if necessary

da = ds[VAR_NAME]
da = da.sel(time=da.time.dt.month == 8)
if da.time.size == 0:
    raise ValueError("No August data found in the model files")
da = da.mean(dim='time', skipna=True)
da = da.load()

lon2d = ds['lonc'].values
lat2d = ds['latc'].values

if lon2d.ndim == 1 and lat2d.ndim == 1:
    lon2d, lat2d = np.meshgrid(lon2d, lat2d)

# ------------------------------------------------------------------
# 3. Remove NaN / Inf cells BEFORE building the tree
# ------------------------------------------------------------------
values = da.values
valid_mask = np.isfinite(values)

lon_flat = lon2d.ravel()[valid_mask.ravel()]
lat_flat = lat2d.ravel()[valid_mask.ravel()]
val_flat = values.ravel()[valid_mask.ravel()]

print(f"Valid model cells after removing NaN/Inf: {len(val_flat)} / {values.size}")

coords = np.column_stack((lon_flat, lat_flat))
tree = cKDTree(coords)

# ------------------------------------------------------------------
# 4. Helper: extract measured + modelled for a dataframe
# ------------------------------------------------------------------
def extract_nearest4(df, value_col='carbon_mg_m2'):
    measured_list = []
    modelled_list = []

    for _, row in df.iterrows():
        lon_o, lat_o = row['lon'], row['lat']
        dists, idxs = tree.query([lon_o, lat_o], k=4)

        vals = val_flat[idxs]
        vals = vals[np.isfinite(vals)]

        if len(vals) > 0:
            measured_list.append(row[value_col])
            modelled_list.append(np.mean(vals))

    return np.asarray(measured_list), np.asarray(modelled_list)

# 2024
meas_2024, mod_2024 = extract_nearest4(df_2024)
print(f'2024 stations with valid nearest-4: {len(meas_2024)}')

# 2014 (only points that fall inside / have valid nearest cells)
meas_2014, mod_2014 = extract_nearest4(df_2014)
print(f'2014 stations with valid nearest-4: {len(meas_2014)}')

# ------------------------------------------------------------------
# 5. Box-plot (three boxes)
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 5))

bp = ax.boxplot(
    [meas_2014, meas_2024, mod_2024],          # 2014 measured, 2024 measured, model
    labels=['Measured\n2014', 'Measured\nSIBES 2024', 'Modelled\n(August, nearest 4)'],
    patch_artist=True,
    showfliers=True,
)

colors = ['#C44E52', '#4C72B0', '#55A868']   # red, blue, green
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Diatom / Phyto carbon (mg C m⁻²)')
ax.set_title('Field data (2014 & 2024) vs. modelled benthic diatom\n(August, nearest 4 cells)')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR / 'boxplot_2014_2024_vs_model_nearest4_August.png', dpi=150)
plt.show()

# ------------------------------------------------------------------
# 6. Quick stats
# ------------------------------------------------------------------
print(f'2014 measured – median: {np.median(meas_2014):.1f}, mean: {np.mean(meas_2014):.1f}')
print(f'2024 measured – median: {np.median(meas_2024):.1f}, mean: {np.mean(meas_2024):.1f}')
print(f'Modelled      – median: {np.median(mod_2024):.1f}, mean: {np.mean(mod_2024):.1f}')

In [ ]:
# to check the measurement date in 2014
